# 02 · Preparación de datos para Looker Studio
**StreamView Analytics · ADY1104 EP3** — continúa desde `01_Limpieza_dataset.ipynb`

Toma `data/processed/netflix_catalog_clean.csv` (Movies + TV Shows ya unidos y limpiados en 00 y 01) y genera las tablas **listas para conectar a Looker Studio**
en `data/processed/looker/`. Looker Studio no sabe interpretar celdas con listas ("Action, Drama"), trata `0` como dato y no ejecuta código, por eso el trabajo pesado se hace aquí.

**Rúbrica:** IE16 (integración y consistencia) · IE18 (KPIs correctamente calculados, con cifras de control) · IE22 (productos organizados).

In [1]:
import pandas as pd, numpy as np, pycountry
from pathlib import Path
pd.set_option('display.max_columns', 40); pd.set_option('display.width', 160)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'data/processed/netflix_catalog_clean.csv').is_file())
SRC  = ROOT/'data/processed/netflix_catalog_clean.csv'
OUT  = ROOT/'data/processed/looker'; OUT.mkdir(parents=True, exist_ok=True)
log = []
def registrar(paso, detalle):
    log.append({'paso':paso,'detalle':detalle}); print(f'✔ {paso}: {detalle}')

df = pd.read_csv(SRC, parse_dates=['date_added'])
print(df.shape); df.head(2)

(32000, 20)


,show_id,type,title,director,cast,country,date_added,release_year,duration,genres,language,description,popularity,vote_count,vote_average,budget,revenue,duration_value,duration_unit,content_key
0,10192,Movie,Shrek Forever After,Mike Mitchell,"Mike Myers, Eddie Murphy, Cameron Diaz, Antoni...",United States of America,2010-05-16,2010,NaN,"Comedy, Adventure, Fantasy, Animation, Family",en,A bored and domesticated Shrek pacts with deal...,203.893,7449,6.380,165000000.0,752600867.0,NaN,NaN,movie_10192_1
1,27205,Movie,Inception,Christopher Nolan,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ken W...","United Kingdom, United States of America",2010-07-15,2010,NaN,"Action, Science Fiction, Adventure",en,"Cobb, a skilled thief who commits corporate es...",156.242,37119,8.369,160000000.0,839030630.0,NaN,NaN,movie_27205_1


## 1. Diagnóstico de la fuente integrada (IE16)

In [2]:
print(df.type.value_counts().to_dict())
print('content_key único:', df.content_key.is_unique, '| show_id repetido entre Movie/TV:', df.show_id.duplicated().sum(), '(por eso se usa content_key como llave)')
cobertura = df.groupby('type')[['director','cast','country','genres','date_added','duration_value','budget','revenue','vote_average']].agg(lambda s: s.notna().mean()).round(2)
cobertura

{'Movie': 16000, 'TV Show': 16000}
content_key único: True | show_id repetido entre Movie/TV: 406 (por eso se usa content_key como llave)


,director,cast,country,genres,date_added,duration_value,budget,revenue,vote_average
type,,,,,,,,,
Movie,0.99,0.99,0.97,0.99,1.0,0.0,1.0,1.0,1.0
TV Show,0.31,0.93,0.89,0.94,1.0,1.0,0.0,0.0,1.0


In [3]:
# Hallazgo importante para el storytelling
coincide = (df.date_added.dt.year == df.release_year).mean()
print(f'date_added.year == release_year en el {coincide:.0%} de los registros')
print('Ceros en budget (películas):', int((df.loc[df.type=='Movie','budget']==0).sum()), '| ceros en revenue:', int((df.loc[df.type=='Movie','revenue']==0).sum()))
print('vote_count = 0:', int((df.vote_count==0).sum()), '| vote_average = 0:', int((df.vote_average==0).sum()))

date_added.year == release_year en el 100% de los registros
Ceros en budget (películas): 11153 | ceros en revenue: 10355
vote_count = 0: 4568 | vote_average = 0: 4572


**Lectura de los hallazgos**
- `date_added` cae **siempre** en el mismo año que `release_year`: no aporta una segunda línea de tiempo. "Contenidos incorporados por año" y "Estrenos por año" serán la misma serie; hay que declararlo en el informe (regla 7). Sí sirve para dar granularidad mensual.
- Presupuesto e ingresos vienen con **0 = dato desconocido** (no un valor real). Si se dejaran, el presupuesto promedio y el ROI quedarían falsamente bajos/imposibles → se convierten a nulo (regla 6).
- `vote_average = 0` con 0 votos significa "sin evaluación", no "mala evaluación".

## 2. Transformaciones

In [4]:
# 2.1 Ceros -> nulos y umbral de coherencia financiera (solo películas; reglas 3, 6 y 9)
UMBRAL = 10_000
mov = df.type.eq('Movie')
ok = mov & df.budget.ge(UMBRAL) & df.revenue.ge(UMBRAL)
df['tiene_datos_financieros'] = ok
df['budget_valido']  = df.budget.where(ok)
df['revenue_valido'] = df.revenue.where(ok)
df['roi']      = df.revenue_valido/df.budget_valido
df['ganancia'] = df.revenue_valido - df.budget_valido
assert df.loc[~mov,'budget_valido'].isna().all()
registrar('Finanzas', f'películas con budget y revenue ≥ US$10.000: {ok.sum():,} de {mov.sum():,}; ROI = revenue/budget')

rangos = [0,1e6,10e6,50e6,100e6,np.inf]; etiq = ['1) < 1 M','2) 1–10 M','3) 10–50 M','4) 50–100 M','5) > 100 M']
df['rango_presupuesto'] = pd.cut(df.budget_valido, rangos, labels=etiq, right=False).astype('object').fillna('Sin dato financiero')
q1 = df.budget_valido.quantile(.25)
df['exito_bajo_presupuesto'] = np.where(ok & df.budget_valido.le(q1) & df.roi.ge(3), 'Sí', 'No')
registrar('Finanzas', f'rango_presupuesto y exito_bajo_presupuesto (presupuesto ≤ Q1 = US${q1:,.0f} y ROI ≥ 3)')

✔ Finanzas: películas con budget y revenue ≥ US$10.000: 3,492 de 16,000; ROI = revenue/budget
✔ Finanzas: rango_presupuesto y exito_bajo_presupuesto (presupuesto ≤ Q1 = US$6,000,000 y ROI ≥ 3)


In [5]:
# 2.2 Valoración: nota 0 sin votos = sin evaluación
sin_votos = df.vote_count.eq(0)
df['vote_average_valida'] = df.vote_average.where(~sin_votos)
df['calificado_num'] = (~sin_votos).astype(int)
df['calificado'] = np.where(sin_votos, 'No', 'Sí')
MIN_VOTOS = 50
df['ranking_confiable'] = np.where(df.vote_count.ge(MIN_VOTOS), 'Sí', 'No')
print('Calificación promedio con ceros: %.2f | sin ceros: %.2f' % (df.vote_average.mean(), df.vote_average_valida.mean()))
registrar('Valoración', f'vote_average_valida (nulo si 0 votos); ranking_confiable = ≥ {MIN_VOTOS} votos')

# Cuadrantes popularidad × calificación (para el scatter de storytelling)
base = df[df.ranking_confiable.eq('Sí')]
mp, mv = base.popularity.median(), base.vote_average_valida.median()
def cuadrante(r):
    if r.ranking_confiable=='No': return 'Sin votos suficientes'
    return ('Alta popularidad' if r.popularity>=mp else 'Baja popularidad')+' · '+('bien evaluado' if r.vote_average_valida>=mv else 'evaluación baja')
df['cuadrante'] = df.apply(cuadrante, axis=1)
print(f'Cortes de cuadrante: popularidad mediana={mp:.1f}, calificación mediana={mv:.2f}'); df.cuadrante.value_counts()

Calificación promedio con ceros: 5.69 | sin ceros: 6.63
✔ Valoración: vote_average_valida (nulo si 0 votos); ranking_confiable = ≥ 50 votos


Cortes de cuadrante: popularidad mediana=14.1, calificación mediana=6.70


cuadrante
Sin votos suficientes                 16603
Alta popularidad · bien evaluado       5136
Baja popularidad · evaluación baja     5091
Baja popularidad · bien evaluado       2607
Alta popularidad · evaluación baja     2563
Name: count, dtype: int64

In [6]:
# 2.3 Campos categóricos: principal, 'Sin dato' explícito y multivaluados
def primero(s):  return s.str.split(',').str[0].str.strip()
df['genero_principal']  = primero(df.genres).fillna('Sin dato')
df['director_principal']= primero(df.director).fillna('Sin dato')
df['pais_principal']    = primero(df.country).fillna('Sin dato')
df['n_generos'] = df.genres.str.count(',').add(1).fillna(0).astype(int)
df['n_paises']  = df.country.str.count(',').add(1).fillna(0).astype(int)
df['coproduccion'] = np.where(df.n_paises>1, 'Sí', 'No')
df['temporadas'] = df.duration_value.where(df.duration_unit.eq('seasons'))
registrar('Categóricas', 'genero/director/pais_principal = primer valor de la lista; nulos -> "Sin dato" (visible como cobertura en el dashboard)')

# Idioma legible
IDIOMAS = {'en':'Inglés','es':'Español','fr':'Francés','de':'Alemán','it':'Italiano','pt':'Portugués','ja':'Japonés','ko':'Coreano','zh':'Chino','cn':'Cantonés',
           'hi':'Hindi','ru':'Ruso','tr':'Turco','th':'Tailandés','ar':'Árabe','sv':'Sueco','da':'Danés','no':'Noruego','nl':'Neerlandés','pl':'Polaco',
           'id':'Indonesio','ta':'Tamil','te':'Telugu','fa':'Persa','xx':'Sin idioma'}
df['idioma'] = df.language.map(IDIOMAS).fillna(df.language.str.upper())

# País -> código ISO-2 (Looker Studio ubica países en el mapa de forma fiable con ISO)
MANUAL = {'Palestinian Territory':'PS','Faeroe Islands':'FO','Guadaloupe':'GP','Russia':'RU',"Cote D'Ivoire":'CI','Macedonia':'MK','US Virgin Islands':'VI',
          'St. Pierre and Miquelon':'PM','St. Kitts and Nevis':'KN','Turkey':'TR','St. Lucia':'LC','Kosovo':'XK'}
def iso(n):
    if n in MANUAL: return MANUAL[n]
    try: return pycountry.countries.lookup(n).alpha_2
    except LookupError: return None
nombres = sorted({x.strip() for s in df.country.dropna() for x in s.split(',')})
ISO = {n: iso(n) for n in nombres}
print('Países sin código ISO (no aparecerán en el mapa):', [n for n,c in ISO.items() if c is None])
df['pais_iso'] = df.pais_principal.map(ISO)

df['decada'] = (df.release_year//10*10).astype(str)+'s'
df['periodo'] = pd.cut(df.release_year, [2009,2014,2019,2025], labels=['2010–2014','2015–2019','2020–2025']).astype(str)
df['fecha'] = df.date_added.dt.strftime('%Y-%m-%d')
df['n'] = 1
registrar('Derivadas', 'idioma, pais_iso, decada, periodo, fecha')

✔ Categóricas: genero/director/pais_principal = primer valor de la lista; nulos -> "Sin dato" (visible como cobertura en el dashboard)
Países sin código ISO (no aparecerán en el mapa): ['Netherlands Antilles']
✔ Derivadas: idioma, pais_iso, decada, periodo, fecha


### 2.4 Tablas *long* (una fila por contenido–entidad)
Necesarias para contar **todos** los géneros, países, actores y directores de cada título, no solo el primero. Cada tabla lleva las dimensiones de filtro (tipo, año) y las métricas,
de modo que cada página de Looker Studio funcione con **una sola fuente** y sus propios controles.

In [7]:
BASE = ['content_key','type','title','release_year','fecha','popularity','vote_average_valida','calificado_num','ranking_confiable']
def largo(col, nuevo):
    t = df.loc[df[col].notna(), BASE].copy()
    t['_v'] = df.loc[df[col].notna(), col].str.split(',')
    t = t.explode('_v'); t[nuevo] = t['_v'].str.strip()
    t = t[t[nuevo].ne('')].drop(columns='_v')
    return t.drop_duplicates(['content_key', nuevo]).reset_index(drop=True)

generos    = largo('genres','genero')
paises     = largo('country','pais'); paises['pais_iso'] = paises.pais.map(ISO)
directores = largo('director','director')
actores    = largo('cast','actor')
for n,t in [('generos',generos),('paises',paises),('directores',directores),('actores',actores)]:
    print(f'{n:11s}{len(t):>9,} filas | {t.iloc[:,-1 if n!="paises" else -2].nunique():>6,} entidades distintas')
registrar('Long', f'generos {len(generos):,} · paises {len(paises):,} · directores {len(directores):,} · actores {len(actores):,} filas')

generos       65,900 filas |     28 entidades distintas
paises        37,636 filas |    147 entidades distintas
directores    24,768 filas | 14,029 entidades distintas
actores      140,394 filas | 59,710 entidades distintas
✔ Long: generos 65,900 · paises 37,636 · directores 24,768 · actores 140,394 filas


## 3. Validaciones

In [8]:
checks = {
 'content_key único': df.content_key.is_unique,
 'series sin datos financieros (regla 3)': df.loc[df.type=='TV Show',['budget_valido','revenue_valido','roi']].isna().all().all(),
 'financieros válidos ≥ umbral': df.loc[df.tiene_datos_financieros,['budget_valido','revenue_valido']].ge(UMBRAL).all().all(),
 'roi = revenue/budget': np.allclose(df.roi.dropna(), (df.revenue_valido/df.budget_valido).dropna()),
 'calificación válida en [0,10]': df.vote_average_valida.dropna().between(0,10).all(),
 'sin nulos en type/title/release_year/popularity': df[['type','title','release_year','popularity']].notna().all().all(),
 'long → content_key existe en la tabla principal': all(t.content_key.isin(df.content_key).all() for t in (generos,paises,directores,actores)),
 'sin listas ("a, b") en dimensiones del dashboard': not df[['genero_principal','pais_principal','director_principal']].apply(lambda s: s.str.contains(',')).any().any(),
}
res = pd.Series(checks, name='OK').to_frame(); display(res); assert all(checks.values())

,OK
content_key único,True
series sin datos financieros (regla 3),True
financieros válidos ≥ umbral,True
roi = revenue/budget,True
"calificación válida en [0,10]",True
sin nulos en type/title/release_year/popularity,True
long → content_key existe en la tabla principal,True
"sin listas (""a, b"") en dimensiones del dashboard",True


## 4. Cifras de control (IE18)
Estos son los valores que deben mostrar los *scorecards* de Looker Studio. **Si no coinciden, hay un error de configuración en el dashboard** (tipo de campo, filtro o métrica).

In [9]:
f = df[df.tiene_datos_financieros]
control = pd.Series({
  'Total de contenidos': len(df),
  'Total de películas': int((df.type=='Movie').sum()),
  'Total de series': int((df.type=='TV Show').sum()),
  'Países representados (todos los listados)': paises.pais.nunique(),
  'Idiomas disponibles': df.language.nunique(),
  'Géneros distintos (todas las etiquetas)': generos.genero.nunique(),
  'Popularidad promedio': round(df.popularity.mean(),2),
  'Popularidad mediana': round(df.popularity.median(),2),
  'Calificación promedio (solo con votos)': round(df.vote_average_valida.mean(),2),
  '% contenidos calificados': round(df.calificado_num.mean()*100,1),
  'Películas con datos financieros': len(f),
  'Presupuesto promedio (US$)': round(f.budget_valido.mean()),
  'Ingresos totales (US$)': int(f.revenue_valido.sum()),
  'ROI agregado (ΣIngresos/ΣPresupuesto)': round(f.revenue_valido.sum()/f.budget_valido.sum(),3),
  'ROI mediano por película': round(f.roi.median(),3),
  'ROI promedio por película': round(f.roi.mean(),3),
}, name='Valor esperado').to_frame(); control

,Valor esperado
Total de contenidos,3.200000e+04
Total de películas,1.600000e+04
Total de series,1.600000e+04
Países representados (todos los listados),1.470000e+02
Idiomas disponibles,8.300000e+01
Géneros distintos (todas las etiquetas),2.800000e+01
Popularidad promedio,4.262000e+01
Popularidad mediana,2.206000e+01
Calificación promedio (solo con votos),6.630000e+00
% contenidos calificados,8.570000e+01


## 5. Exportación (IE22)

In [10]:
COLS = ['content_key','type','title','director_principal','genero_principal','pais_principal','pais_iso','idioma','release_year','fecha','decada','periodo',
        'popularity','vote_count','vote_average_valida','calificado','calificado_num','ranking_confiable','cuadrante','temporadas','n_generos','n_paises','coproduccion',
        'budget_valido','revenue_valido','roi','ganancia','tiene_datos_financieros','rango_presupuesto','exito_bajo_presupuesto','n']
principal = df[COLS].copy()
principal['tiene_datos_financieros'] = np.where(principal.tiene_datos_financieros,'Sí','No')

principal.to_csv(OUT/'catalogo_looker.csv', index=False, encoding='utf-8')
generos.to_csv(OUT/'generos_long.csv', index=False);       paises.to_csv(OUT/'paises_long.csv', index=False)
directores.to_csv(OUT/'directores_long.csv', index=False); actores.to_csv(OUT/'actores_long.csv', index=False)
control.to_csv(OUT/'cifras_de_control.csv')
pd.DataFrame(log).to_csv(OUT/'registro_transformaciones.csv', index=False)

# Diccionario con el tipo que debe tener cada campo en Looker Studio
tipo = {'content_key':'Texto','type':'Texto','title':'Texto','director_principal':'Texto','genero_principal':'Texto','pais_principal':'Texto','pais_iso':'Geo > País (código ISO 3166-1 alfa-2)',
 'idioma':'Texto','release_year':'Número (usar como dimensión; formato sin separador de miles)','fecha':'Fecha (AAAA-MM-DD)','decada':'Texto','periodo':'Texto',
 'popularity':'Número','vote_count':'Número','vote_average_valida':'Número (agregación: Promedio)','calificado':'Texto','calificado_num':'Número (Promedio = % calificados)',
 'ranking_confiable':'Texto','cuadrante':'Texto','temporadas':'Número','n_generos':'Número','n_paises':'Número','coproduccion':'Texto',
 'budget_valido':'Moneda (USD)','revenue_valido':'Moneda (USD)','roi':'Número (usar Mediana)','ganancia':'Moneda (USD)','tiene_datos_financieros':'Texto',
 'rango_presupuesto':'Texto','exito_bajo_presupuesto':'Texto','n':'Número (Suma = conteo de contenidos)','genero':'Texto','pais':'Texto','director':'Texto','actor':'Texto'}
desc = {'content_key':'Llave única (show_id colisiona entre películas y series)','fecha':'date_added; su año coincide 100 % con release_year',
 'vote_average_valida':'Nota TMDB 0–10; nulo si no hay votos','ranking_confiable':'Sí si vote_count ≥ 50 (para rankings)','cuadrante':'Popularidad × calificación vs medianas (solo títulos con ≥ 50 votos)',
 'budget_valido':'Solo películas con presupuesto e ingresos ≥ US$10.000','roi':'revenue_valido / budget_valido','pais_iso':'Para mapa coroplético','n':'Constante 1 para sumar'}
dic = pd.DataFrame({'campo':list(tipo),'tipo_en_looker':list(tipo.values()),'descripcion':[desc.get(k,'') for k in tipo]})
dic.to_csv(OUT/'diccionario_campos_looker.csv', index=False)
print({p.name: f'{p.stat().st_size/1e6:.1f} MB' for p in sorted(OUT.glob('*.csv'))})

{'actores_long.csv': '12.2 MB', 'catalogo_looker.csv': '6.9 MB', 'cifras_de_control.csv': '0.0 MB', 'diccionario_campos_looker.csv': '0.0 MB', 'directores_long.csv': '2.1 MB', 'generos_long.csv': '5.4 MB', 'paises_long.csv': '3.3 MB', 'registro_transformaciones.csv': '0.0 MB'}


## 6. Qué documentar en el informe
1. Se trabajó con **una sola llave (`content_key`)** porque los `show_id` se repiten entre películas y series.
2. **Ceros → nulos** en presupuesto/ingresos y umbral de US$ 10.000; sin esto el ROI carece de sentido.
3. **`vote_average = 0` sin votos** excluido de promedios; rankings con ≥ 50 votos.
4. **Multivaluados** en tablas *long* para conteos correctos de géneros, países, directores y actores.
5. **Limitaciones:** `date_added` = año de estreno (una sola línea de tiempo); el extracto es de 2.000 títulos por año (evolución anual no refleja producción real); `duration` solo existe como nº de temporadas en series; `popularity` es un índice relativo (regla 5).

**Siguiente paso:** `dashboard/guia_looker_studio.md`.